In [ ]:
%pip install peft evaluate
%pip install -U torchao

In [2]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU device: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU device: NVIDIA L4


In [3]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")
except:
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception as e:
        print(e)

In [ ]:
# from huggingface_hub import sync_bucket

# sync_bucket(
#     "hf://buckets/RobChio/swissgerman",
#     "./bucket/"
# )

In [50]:
labels = {
    0: "AG",
    1: "BE",
    2: "BS",
    3: "GR",
    4: "LU",
    5: "SG",
    6: "VS",
    7: "ZH",
}

id2label = labels
label2id = {v: k for k, v in labels.items()}
num_labels = len(labels)

In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperForAudioClassification, WhisperProcessor, WhisperFeatureExtractor
import torch

#model_id = "Flix-AI/flix-swissgerman-full"
model_id = "openai/whisper-tiny"

feature_extractor = WhisperFeatureExtractor.from_pretrained(model_id)

model = WhisperForAudioClassification.from_pretrained(
    model_id,
    num_labels=8,
    label2id=label2id,
    id2label=id2label,
    torch_dtype=torch.bfloat16,
    apply_spec_augment=True, # TODO: verify specaugment params
    mask_time_prob=0.05,
    # mask_time_length=10,
    mask_feature_prob=0.05,
    # mask_feature_length=10,
)

In [20]:
from peft import PeftModel, LoraConfig, get_peft_model, TaskType

peft_config = LoraConfig(
    r=160,
    lora_alpha=32,
    # target_modules=["q_proj", "v_proj"],
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"],
    # target_modules=[
    # "q_proj",
    # "k_proj",
    # "v_proj",
    # "out_proj",
    # ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_CLS,
    modules_to_save=["projector", "classifier"],
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 4,425,736 || all params: 12,734,736 || trainable%: 34.7533


In [ ]:
import datasets
from datasets import load_dataset, load_from_disk, Audio, interleave_datasets, concatenate_datasets, Value

# Training data: Combine SwissDial and ArchiMob
ds_swissdial = load_dataset("RobChio/swiss-dial-preprocessed", split="train")
ds_archimob = load_dataset("RobChio/archimob-preprocessed", split="train")

ds_swissdial = ds_swissdial.cast_column("labels", Value(dtype="int64")) # fix type mismatch

ds_swissdial.set_format(type="torch", columns=["audio", "labels"])
ds_archimob.set_format(type="torch", columns=["audio", "labels"])

combined_train = concatenate_datasets([ds_swissdial, ds_archimob])
train_dataset = combined_train.cast_column("audio", Audio(sampling_rate=16000))
train_dataset = train_dataset.shuffle(seed=42)
train_dataset.set_format(type="torch", columns=["audio", "labels"])

# Validation data from All Swiss German Dialect Test Set
ds_asgdts = load_dataset("RobChio/asgdts-preprocessed", split="test", streaming=False)
val_dataset = ds_asgdts # TODO: quick val and full val?
val_dataset.set_format(type="torch", columns=["audio", "labels"])
val_dataset = val_dataset.cast_column("audio", Audio(sampling_rate=16000))

In [ ]:
def preprocess_function(batch):
    audios = [x["array"] for x in batch["audio"]]

    inputs = feature_extractor(
        audios,
        sampling_rate=16000,
        return_tensors="np",
        # max_length=160000,  # Truncate at 10 seconds (16,000 Hz * 10s)
        # truncation=True,
    )

    return {
        "input_features": inputs.input_features,
        "labels": batch["labels"],
    }

# Apply mapping (audio is loaded on the fly)
train_dataset_preprocessed = train_dataset.map(
    preprocess_function,
    remove_columns=train_dataset.column_names,
    batched=True,
    batch_size=8,
    num_proc=1,
)

val_dataset_preprocessed = val_dataset.map(
    preprocess_function,
    remove_columns=["audio"],
    batched=True,
    batch_size=64,
    num_proc=8,
)

In [ ]:
from torch.nn.utils.rnn import pad_sequence

class FastWhisperDataCollator:
    def __call__(self, features):
        tensors = [
            torch.as_tensor(f["input_features"], dtype=torch.float32)
            for f in features
        ]
        # Dynamic CPU padding to the longest sequence in the current batch
        input_features = pad_sequence(tensors, batch_first=True, padding_value=0.0)
        labels = torch.tensor([f["labels"] for f in features], dtype=torch.long)
        return {"input_features": input_features, "labels": labels}

data_collator = FastWhisperDataCollator()

In [42]:
from torch.nn.utils.rnn import pad_sequence

class FastWhisperDataCollator:
    def __call__(self, features):
        raw_audio = [f["audio"]["array"] for f in features]

        inputs = feature_extractor(
            raw_audio,
            sampling_rate=16000,
            return_tensors="pt"
        )

        #input_features = pad_sequence(inputs, batch_first=True, padding_value=0.0)
        input_features = inputs.input_features
        labels = torch.tensor([f["labels"] for f in features], dtype=torch.long)
        return {"input_features": input_features, "labels": labels}

data_collator = FastWhisperDataCollator()

In [32]:
import torch
from torch import nn
from transformers import Trainer
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        if class_weights is not None and not isinstance(
            class_weights, torch.Tensor
        ):
            self.class_weights = torch.tensor(class_weights, dtype=torch.float32)
        else:
            self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        smoothing = getattr(self.args, "label_smoothing_factor", 0.0)

        if self.class_weights is not None:
            weights = self.class_weights.to(logits.device)
            loss_fct = nn.CrossEntropyLoss(weight=weights, label_smoothing=smoothing)
        else:
            loss_fct = nn.CrossEntropyLoss(label_smoothing=smoothing)

        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

In [35]:
train_labels = train_dataset.with_format("numpy")["labels"]

class_weights = compute_class_weight(
    class_weight="balanced", classes=np.arange(8), y=train_labels
)

# Dampen class weight extremity using square root scaling
# class_weights = class_weights**0.5

In [55]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
import evaluate
import numpy as np
import torch.nn.functional as F

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    predictions = np.argmax(logits, axis=-1)

    macro_f1 = f1.compute(predictions=predictions, references=labels, average="macro")["f1"]
    acc = accuracy.compute(predictions=predictions, references=labels)["accuracy"]
    return {"f1": macro_f1, "accuracy": acc}

training_args = TrainingArguments(
    output_dir="./swissgerman-dialect-classifier",
    # eval_strategy="epoch",
    # save_strategy="epoch",
    eval_strategy="steps",
    save_strategy="steps",
    eval_steps=300,
    save_steps=300,
    learning_rate=1e-5,
    #max_grad_norm=0.5, # gradient clipping
    weight_decay=0.03,
    label_smoothing_factor=0.1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=4, # 256
    num_train_epochs=2,
    warmup_steps=250,
    lr_scheduler_type="cosine",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1", #"accuracy"
    greater_is_better=True,
    bf16=True,
    remove_unused_columns=False,
    dataloader_num_workers=4, # multiple workers with streaming causes data duplication
    dataloader_pin_memory=True,
    dataloader_persistent_workers=True,
    dataloader_prefetch_factor=2,
    torch_compile=False,
    dataloader_drop_last=False,
    push_to_hub=False,
    hub_model_id="RobChio/swissgerman-dialect-classifier",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

In [56]:
# Train model
train_result = trainer.train()

# Log and save training state and metrics
# trainer.log_metrics("train", train_result.metrics)
# trainer.save_metrics("train", train_result.metrics)
# trainer.save_state()

Step,Training Loss,Validation Loss,F1,Accuracy
300,2.255018,2.986735,0.115377,0.141565
600,2.308438,3.028114,0.115424,0.142435
900,2.312139,3.024796,0.116994,0.144000
1200,2.264483,3.024717,0.117234,0.144522
1272,2.237368,3.024730,0.117147,0.144348


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
trainer.push_to_hub(commit_message="Pretrained classifier")

In [ ]:
# eval_metrics = trainer.evaluate(eval_dataset=test_dataset)
# trainer.log_metrics("eval", eval_metrics)
# trainer.save_metrics("eval", eval_metrics)

In [ ]:
# from google.colab import runtime
# runtime.unassign()